<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **ACCESS to TEMPO data via OPeNDAP**

 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)
2. <font size="3"> **Optional**: If running notebook locally, install the conda environment in `environment.yml` file and install conda environment to run notebook.


 <span style='color:#ff6666'><font size="5"> **Objectives**
### Basics
- <font size="3"> Brief Introduction to <span style='color:#ff6666'>**OPeNDAP**</span> (i.e. **dap2** vs **dap4**). 
- <font size="3"> How to find <span style='color:#ff6666'>**OPeNDAP**</span> URLs.
- <font size="3"> Inspecting metadata (differences between **browser** / <span style='color:#ff6666'>**pydap**</span> and **Xarray**).
- <font size="3"> **Naive approach**: access data from a url using **Xarray** + <span style='color:#ff6666'>**pydap**<span style='color:black'>. Here we demonstrate different ways to authenticate.

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3"> **a)** Naive approaches.
- <font size="3"> **b)** Streaming data


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess

# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

In [ ]:
TEMPO_O3TOT_L3_V03_ccid = "C2930764281-LARC_CLOUD"
time_range = [dt.datetime(2025, 9, 1), dt.datetime(2025, 9, 30)] # One month of data

cmr_urls = get_cmr_urls(ccid=TEMPO_O3TOT_L3_V03_ccid, time_range=time_range, limit=1000) # you can incread the limit of results


print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[0]

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via OPeNDAP**

<font size="3.5"> You can authenticate via:

* <font size="3.5"> `.netrc` file (username password)
* <font size="3.5"> Token bearer header


<font size="3.5"> OPeNDAP's Hyrax server support both forms of authentication. Below we demonstrate using earthaccess to store and inherit EDL credentials into a session that will be used to stream data from OPeNDAP in the Cloud.


In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Server-Side Subsetting**

### Subset by Variable name

<font size="3.5"> To subset by variable name, we first need to know what variables there are in the remote file. Below we access the metadata-only of the remote file via PyDAP, reusing the session object.

<font size="3.5"> Below we request the DAP4 (hierarchial) metadata from the remote server. To specify the protocol, we have 2 options:

1. `url.replace("https","dap4")`. <font size="3.5"> This replaces the `https` with `dap4`. "dap4" is not a separate protocol from https, it is simply a way to tell the client (PyDAP or Xarray) which type of metadata request to send the server.
2. <font size="3.5"> With PyDAP one can also define the argument `"protocol="dap4"` when opening a url. 

<font size="3.5"> Below we follow option #1) above.


In [ ]:
%%time
pyds = open_url(cmr_urls[0].replace("https","dap4"), session=my_session)
pyds.tree()

<font size="3.5"> Having now the full list of variables, we are interested in:
* <font size="3.5"> Geographic location data,
* <font size="3.5"> The variable `weight`.
* <font size="3.5"> Time.
* <font size="3.5"> All of the data inside the group `support_data`
* <font size="3.5"> The variable `o3_below_cloud` for further processing.


<font size="3.5"> The remote file contains hierarchies such as Groups, which act like folder in a filesystem. In DAP4 we can define variables across different hierarchies, by defining their `fully qualifying name`, analogous to a file inside a filesystem. 

<font size="3.5"> The list of variables we are interested in are declared in `keep_variables`, below


In [ ]:
keep_variables = ['/longitude', '/latitude', '/time', "/support_data", "/product/o3_below_cloud"]


### Subset by coordinate values

<font size="3.5"> Below we use Xarray to download coordinate data using PyDAP in the background. THe goal is to identify a region of interest and only download data inside that region of interest.



In [ ]:
# define bounding box by edges
lat_min, lat_max = 45, 60
lon_min, lon_max = -123, -120.5

ds = xr.open_dataset(cmr_urls[0].replace("https","dap4"), session=my_session, engine='pydap')

lat_index = ds.latitude.to_index()
lon_index = ds.longitude.to_index()

lat_start = lat_index.get_indexer([lat_min], method="nearest")[0]
lat_end   = lat_index.get_indexer([lat_max], method="nearest")[0]

lon_start = lon_index.get_indexer([lon_min], method="nearest")[0]
lon_end   = lon_index.get_indexer([lon_max], method="nearest")[0]

# define slicing argument to use in PyDAP to stream the subset of data
dim_slices = {'latitude': (lat_start, lat_end), 'longitude': (lon_start, lon_end)}


### Stream data 

In [ ]:
%%time
dap_to_netcdf(cmr_urls[:30], session=my_session, output_path = "./data", dim_slices=dim_slices, keep_variables=keep_variables)

## check the data

In [ ]:
%%time
ds1 = xr.open_mfdataset("data/TEMPO_O3*.nc4", group="/", parallel=True)
ds2 = xr.open_mfdataset("data/TEMPO_O3*.nc4", group="/support_data", parallel=True, concat_dim='time', combine='nested')
ds3 = xr.open_mfdataset("data/TEMPO_O3*.nc4", group="/product", parallel=True, concat_dim='time', combine='nested')
ds = xr.merge([ds1, ds2, ds3])

In [ ]:
ds

In [ ]:
ds['terrain_height'].isel(time=0).plot();